06 Task-Aware Evaluation

Goal: evaluate pin quality beyond median offset. This notebook introduces movable-place evaluation, arrival-cost scoring, and optional comparison for future repositioning methods.

In [2]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features import place_complexity, pin_ambiguity, should_move_rule
from src.metrics import (
    haversine_meters,
    task_aware_report,
    segmented_task_report,
    arrival_cost_score,
)

PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT


WindowsPath('c:/Users/aaron/Documents/Pin-To-Place')

In [3]:
combined_path = PROCESSED / "ground_truth_combined.csv"

if combined_path.exists():
    df = pd.read_csv(combined_path)
else:
    files = sorted(PROCESSED.glob("ground_truth_*.csv"))
    df = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)

df["place_complexity"] = df.get("place_complexity", df.apply(place_complexity, axis=1))
df["pin_ambiguity"] = df.get("pin_ambiguity", df.apply(pin_ambiguity, axis=1))
df["should_move"] = df.get("should_move", df.apply(should_move_rule, axis=1))

df.shape


(3425, 20)

In [4]:
baseline_report = task_aware_report(df)

baseline_report


{'count': 3425,
 'mean_m': np.float64(6.4),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(37.55),
 'p95_m': np.float64(40.27),
 'max_m': np.float64(88.56),
 'pct_exact_no_move': np.float64(79.6),
 'pct_over_10m': np.float64(19.2),
 'pct_over_25m': np.float64(14.0),
 'pct_over_50m': np.float64(0.1)}

In [5]:
movable_df = df[df["should_move"]].copy()
protected_df = df[~df["should_move"]].copy()

{
    "all_rows": len(df),
    "movable_rows": len(movable_df),
    "protected_rows": len(protected_df),
    "movable_pct": round(len(movable_df) / len(df) * 100, 1),
}


{'all_rows': 3425,
 'movable_rows': 675,
 'protected_rows': 2750,
 'movable_pct': 19.7}

In [6]:
{
    "all_places": task_aware_report(df),
    "movable_places": task_aware_report(movable_df) if len(movable_df) else {},
    "protected_places": task_aware_report(protected_df) if len(protected_df) else {},
}


{'all_places': {'count': 3425,
  'mean_m': np.float64(6.4),
  'median_m': np.float64(0.0),
  'p90_m': np.float64(37.55),
  'p95_m': np.float64(40.27),
  'max_m': np.float64(88.56),
  'pct_exact_no_move': np.float64(79.6),
  'pct_over_10m': np.float64(19.2),
  'pct_over_25m': np.float64(14.0),
  'pct_over_50m': np.float64(0.1)},
 'movable_places': {'count': 675,
  'mean_m': np.float64(32.14),
  'median_m': np.float64(37.64),
  'p90_m': np.float64(42.49),
  'p95_m': np.float64(43.42),
  'max_m': np.float64(88.56),
  'pct_exact_no_move': np.float64(0.0),
  'pct_over_10m': np.float64(97.6),
  'pct_over_25m': np.float64(71.0),
  'pct_over_50m': np.float64(0.3)},
 'protected_places': {'count': 2750,
  'mean_m': np.float64(0.08),
  'median_m': np.float64(0.0),
  'p90_m': np.float64(0.0),
  'p95_m': np.float64(0.0),
  'max_m': np.float64(10.0),
  'pct_exact_no_move': np.float64(99.2),
  'pct_over_10m': np.float64(0.0),
  'pct_over_25m': np.float64(0.0),
  'pct_over_50m': np.float64(0.0)}}

In [7]:
segmented_task_report(df, "tier_label")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,209,15.88,8.47,39.70,41.17,78.73,48.3,44.0,37.3,0.5,open_space
3,2307,7.87,0.00,38.45,40.85,47.39,74.9,24.1,17.0,0.0,standard_commercial
0,163,2.69,0.00,0.00,33.15,88.56,93.3,6.7,5.5,0.6,multi_tenant
1,746,0.00,0.00,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [8]:
segmented_task_report(df, "place_complexity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
0,303,11.67,0.0,39.05,41.02,46.88,61.7,34.3,26.7,0.0,complex
2,3000,5.86,0.0,37.27,40.20,88.56,81.4,17.7,12.7,0.1,simple
1,122,6.61,0.0,38.24,39.84,43.23,80.3,19.7,14.8,0.0,multi_tenant


In [9]:
segmented_task_report(df, "pin_ambiguity")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
1,2064,7.96,0.0,38.51,40.90,88.56,75.0,23.8,17.4,0.1,low
2,331,5.87,0.0,37.02,39.77,47.39,80.1,19.0,11.8,0.0,medium
0,1030,3.43,0.0,10.12,36.81,46.88,88.7,10.1,7.9,0.0,high


In [10]:
def infer_arrival_friction(row) -> dict:
    """
    V1 heuristic arrival-friction labels.

    These are intentionally conservative placeholders until you have
    sidewalk, curb-cut, road-network, parking-lot, or imagery-derived features.
    """
    tier = row.get("tier_label")
    complexity = row.get("place_complexity")
    category = str(row.get("category_primary", "")).lower()

    parking_lot_crossing = tier in {"standard_commercial", "multi_tenant"} and complexity in {
        "complex",
        "multi_tenant",
    }

    sidewalk_visible = None
    barrier_detected = False

    if tier == "open_space":
        sidewalk_visible = False

    if category in {"campground", "rv_park", "resort"}:
        parking_lot_crossing = True

    return {
        "sidewalk_visible": sidewalk_visible,
        "parking_lot_crossing": parking_lot_crossing,
        "barrier_detected": barrier_detected,
    }


friction = df.apply(infer_arrival_friction, axis=1, result_type="expand")
df = pd.concat([df, friction], axis=1)

df[["sidewalk_visible", "parking_lot_crossing", "barrier_detected"]].head()

,sidewalk_visible,parking_lot_crossing,barrier_detected
0,None,False,False
1,None,False,False
2,None,False,False
3,None,False,False
4,None,False,False


In [11]:
df["arrival_cost_m"] = df.apply(
    lambda row: arrival_cost_score(
        distance_m=row["offset_haversine_m"],
        sidewalk_visible=row["sidewalk_visible"],
        parking_lot_crossing=row["parking_lot_crossing"],
        barrier_detected=row["barrier_detected"],
    ),
    axis=1,
)

task_aware_report(df, offset_col="arrival_cost_m")

{'count': 3425,
 'mean_m': np.float64(8.35),
 'median_m': np.float64(0.0),
 'p90_m': np.float64(38.37),
 'p95_m': np.float64(42.61),
 'max_m': np.float64(93.73),
 'pct_exact_no_move': np.float64(71.4),
 'pct_over_10m': np.float64(22.8),
 'pct_over_25m': np.float64(15.0),
 'pct_over_50m': np.float64(2.8)}

In [12]:
segmented_task_report(df, "tier_label", offset_col="arrival_cost_m")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
2,209,35.91,25.0,63.66,65.44,93.73,0.0,100.0,47.4,36.8,open_space
3,2307,8.82,0.0,38.86,41.43,56.88,68.4,24.3,17.6,0.7,standard_commercial
0,163,4.60,0.0,10.00,33.15,88.56,74.8,6.7,5.5,0.6,multi_tenant
1,746,0.00,0.0,0.00,0.00,0.00,100.0,0.0,0.0,0.0,no_building


In [13]:
segmented_task_report(df, "place_complexity", offset_col="arrival_cost_m")

,count,mean_m,median_m,p90_m,p95_m,max_m,pct_exact_no_move,pct_over_10m,pct_over_25m,pct_over_50m,segment
0,303,26.31,19.62,61.74,63.98,69.44,6.3,51.5,35.0,21.1,complex
1,122,12.97,10.00,48.24,49.84,53.23,36.9,20.5,18.9,4.9,multi_tenant
2,3000,6.35,0.00,37.45,40.52,93.73,79.4,20.0,12.8,0.8,simple


In [14]:
def compute_method_offset(df, lat_col, lon_col, output_col):
    result = df.copy()

    valid = result[[lat_col, lon_col, "gt_lat", "gt_lon"]].notna().all(axis=1)

    result[output_col] = np.nan
    result.loc[valid, output_col] = result.loc[valid].apply(
        lambda row: haversine_meters(
            row[lat_col],
            row[lon_col],
            row["gt_lat"],
            row["gt_lon"],
        ),
        axis=1,
    )

    return result


candidate_methods = {
    "ensemble": ("ensemble_lat", "ensemble_lon"),
    "ranker": ("ranker_lat", "ranker_lon"),
    "llm": ("llm_lat", "llm_lon"),
}

available_methods = {
    name: cols
    for name, cols in candidate_methods.items()
    if cols[0] in df.columns and cols[1] in df.columns
}

available_methods

{}

In [15]:
method_reports = []

for method_name, (lat_col, lon_col) in available_methods.items():
    offset_col = f"{method_name}_offset_m"
    df = compute_method_offset(df, lat_col, lon_col, offset_col)

    baseline = df["offset_haversine_m"]
    method = df[offset_col]

    valid = method.notna()
    regression_rate = round((method[valid] > baseline[valid]).mean() * 100, 2)

    report = task_aware_report(df[valid], offset_col=offset_col)
    report["method"] = method_name
    report["regression_rate_pct"] = regression_rate
    method_reports.append(report)

if method_reports:
    pd.DataFrame(method_reports).sort_values("p95_m")
else:
    "No repositioning method columns found yet. This is expected until ensemble/ranker/LLM outputs are generated."

In [16]:
evaluation_cols = [
    "id",
    "name",
    "category_primary",
    "region",
    "tier_label",
    "place_complexity",
    "pin_ambiguity",
    "should_move",
    "offset_haversine_m",
    "arrival_cost_m",
    "gt_confidence",
    "sidewalk_visible",
    "parking_lot_crossing",
    "barrier_detected",
]

task_eval = df[evaluation_cols].copy()
task_eval.to_csv(PROCESSED / "task_aware_evaluation.csv", index=False)

task_eval.sort_values("arrival_cost_m", ascending=False).head(50)

,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,should_move,offset_haversine_m,arrival_cost_m,gt_confidence,sidewalk_visible,parking_lot_crossing,barrier_detected
3387,08f2ad20087595a6033fe3dd7fb34603,Prestige Funeral Home,funeral_services_and_cemeteries,SC,open_space,simple,low,True,78.728265,93.73,0.80,False,False,False
1414,08f2ad3c69044aca03c94926717c3b4a,Redbox,rental_kiosks,NC,multi_tenant,simple,low,True,88.558515,88.56,0.90,None,False,False
2170,08f4649589d05a4003d89cadf70517a8,Bellows Camp Site,campground,NaN,open_space,complex,high,True,44.438610,69.44,0.90,False,True,False
2639,08f44f44e8adc34203fbe6ae002a8dff,Lake Kerr Rentals,rv_park,FL,open_space,complex,high,True,42.870552,67.87,0.90,False,True,False
2305,08f441305256450603596e4dfefc80fb,Sun Resorts & Residences Ft. Myers Beach,campground,FL,open_space,complex,high,True,42.709897,67.71,0.80,False,True,False
3247,08f489562c40cc9e031a6145e6b8090b,Creekside Rv Village,rv_park,TX,open_space,complex,high,True,42.117950,67.12,0.90,False,True,False
2117,08f489c85a292422038343eb5e9cb8bc,Medina River RV Park,rv_park,TX,open_space,complex,high,True,41.603770,66.60,0.80,False,True,False
2820,08f44f082e49c142032785383719bd54,Starke / Gainesville N.E. KOA Holiday,campground,FL,open_space,complex,high,True,41.354954,66.35,0.90,False,True,False
2080,08f48981360c349403c80ceff2683b6b,Hofbrau RV Park,rv_park,TX,open_space,complex,high,True,41.133176,66.13,0.90,False,True,False
1341,08f4468143b34d0d03cca189ab50c32b,Arrowhead RV Park,rv_park,TX,open_space,complex,high,True,41.016142,66.02,0.90,False,True,False


In [17]:
summary_lines = []

summary_lines.append("Task-Aware Evaluation Summary")
summary_lines.append("")
summary_lines.append("Baseline geometric offset")
summary_lines.append(str(task_aware_report(df, offset_col="offset_haversine_m")))
summary_lines.append("")
summary_lines.append("Arrival-cost score")
summary_lines.append(str(task_aware_report(df, offset_col="arrival_cost_m")))
summary_lines.append("")
summary_lines.append("Arrival cost by tier")
summary_lines.append(segmented_task_report(df, "tier_label", offset_col="arrival_cost_m").to_string(index=False))
summary_lines.append("")
summary_lines.append("Arrival cost by complexity")
summary_lines.append(segmented_task_report(df, "place_complexity", offset_col="arrival_cost_m").to_string(index=False))

summary_path = PROCESSED / "task_aware_evaluation_summary.txt"
summary_path.write_text("\n".join(summary_lines))

summary_path

WindowsPath('c:/Users/aaron/Documents/Pin-To-Place/data/processed/task_aware_evaluation_summary.txt')

First, place_complexity separates simple places from multi-tenant and complex places. This makes the analysis fairer because a campground, resort, or shopping center should not be judged the same way as a standalone pizza shop.

Second, pin_ambiguity marks whether a place has one obvious pin or several plausible targets. This gives the project a more research-like angle: sometimes the problem is not “bad coordinate,” it is “the coordinate is inherently ambiguous.”

Third, should_move turns repositioning into a conservative decision. Since your current median offset is already 0.0m, the model should first decide whether a pin deserves movement at all.

Fourth, arrival_cost_m starts moving the project beyond raw distance. It is a v1 score for practical arrival friction: not just “how far is the pin from the label,” but “does this pin make arrival harder?”